**ELT**

**0. Importando bibliotecas**

In [ ]:
# Instalar se não tiver instalado:
# pip install pandas
# pip install sqlalchemy
# pip install psycopg2-binary

In [3]:
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text

**1. Extraindo e juntando os dados**

Aqui os arquivos são carregados e os dados são colocados em DataFrames do pandas.

Antes de rodar é necessario extrair os arquivos "*.rar" da pasta "Dados" primeiro.

In [5]:
# Caminho dos arquivos com os dados
arquivos_path = Path("..") / "Datas"
arquivos_name_list = sorted([arquivo.name for arquivo in arquivos_path.glob("*.csv")])

# Carrega os arquivos como DataFrames do pandas
# Mantemos tudo como texto na leitura para preservar CNPJs e padronizar a limpeza depois.
tdf_dict = {}
for arquivo_name in arquivos_name_list:
    try:
        path = arquivos_path / arquivo_name
        tdf = pd.read_csv(path, sep=";", encoding="cp1252", dtype=str)
        tdf["arquivo_origem"] = arquivo_name
        tdf_dict[arquivo_name] = tdf
    except Exception as e:
        print(f"Erro ao ler o arquivo '{arquivo_name}': {e}")

# Junta os dados em um unico DataFrame:
df = pd.concat(tdf_dict.values(), ignore_index=True)

Foram utilizados os dados de 2024 a 2026. O arquivo de 2026 ainda pode estar em atualização, então vale conferir o volume antes de usar comparações finais.

**3. Enviando dados para o PostgreSQL**

Primeiro é necessario preencher os campos a suas informações do Postgre.

In [7]:
usuario_db = 'postgres'      # É o usuário padrão criado pelo instalador
senha_db = 'nic123'          # A senha que você digitou DURANTE a instalação
host_db = 'localhost'        # Significa "esta minha máquina local"
porta_db = '5432'            # A porta padrão do PostgreSQL
nome_banco_db = 'postgres'   # O banco de dados inicial que o instalador cria automaticamente


Depois é feita a conexão com o banco de dados padrão do postgre para verificar se o banco de dados informado existe ou não. Cria o banco de dados caso ele não exista.

In [8]:
# Criando a string de conexao com banco de dados padrão do postgre
DATABASE_PADRAO_URL = f'postgresql://{usuario_db}:{senha_db}@{host_db}:{porta_db}/postgres?client_encoding=utf8'

# Conecta ao banco padrão postgres
engine_padrao = create_engine(DATABASE_PADRAO_URL)

# Verifica se banco de dados já existe ou não:
with engine_padrao.connect() as con:
    existe = con.execute(
        text(
            "SELECT 1 FROM pg_database WHERE datname = :nome"
        ),
        {"nome": nome_banco_db}
    ).scalar()

    # Cria banco de dados caso não exista:
    if not existe:
        con.execute(text("COMMIT"))
        con.execute(text(f'CREATE DATABASE "{nome_banco_db}"'))
        print(f"Banco {nome_banco_db} criado.")
    else:
        print(f"Banco {nome_banco_db} já existe.")

Banco postgres já existe.


Por fim, estabelece Conecxão com o banco de dados e envia os dados que foram extraidos e concatenados para o Postgre.

In [10]:
print("Tentando criar conexão com o banco PostgreSQL...")

# Criando a string de conexao com banco de dados indicado
DATABASE_URL = f'postgresql://{usuario_db}:{senha_db}@{host_db}:{porta_db}/{nome_banco_db}?client_encoding=utf8'

try:
    # Estabelecendo conexão ativa
    engine = create_engine(DATABASE_URL)
    print("Conexao bem-sucedida! Iniciando a carga dos dados no Postgres...\n")
    
    # Carregando dados no Postgres
    
    print("Carregando Dados...")
    df.to_sql('Dados_Bronze', con=engine, if_exists='replace', index=False)
    

    print("\n CARGA CONCLUÍDA! Uma tabela para os dados foi criada no seu PostgreSQL.")

except Exception as e:
    print(f"\n--- [ERRO CRÍTICO] ---")
    print(f"Erro ao conectar ou carregar os dados no PostgreSQL: {e}")

Tentando criar conexão com o banco PostgreSQL...
Conexao bem-sucedida! Iniciando a carga dos dados no Postgres...

Carregando Dados...

 CARGA CONCLUÍDA! Uma tabela para os dados foi criada no seu PostgreSQL.


Após a execução desse notebook, execute o arquivo "tratamento.sql" que ira fazer o tratamento dos dados enviados para o postgre.